# Test: Claude Sonnet — Validator V4 (API)

Anthropic API-based. Default validator alongside V1 and V2.

**Prerequisites:** `ANTHROPIC_API_KEY` set in environment or `.env`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / '.env')

from models.utils import ClaudeSonnet

model = ClaudeSonnet()
print('Model config:')
model.get_config()

## 1. Health Check

In [ ]:
assert model.health_check(), 'API key invalid or API unreachable!'
print('Health check passed — API key valid')

## 2. Chat Completions

In [ ]:
messages = [
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 2a. Basic chat
resp = model.chat(messages, system='You are a SOC analyst. Be concise.', max_tokens=100)
print('Basic chat:', resp.content[0].text)

In [ ]:
# 2b. System message extracted from messages list
msgs_with_system = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]
resp = model.chat(msgs_with_system, max_tokens=100)
print('System extracted:', resp.content[0].text)

In [ ]:
# 2c. Deterministic
resp = model.chat_deterministic(messages, system='You are a SOC analyst.', max_tokens=100)
print('Deterministic:', resp.content[0].text)

In [ ]:
# 2d. Creative
resp = model.chat_creative(messages, system='You are a SOC analyst.', max_tokens=100)
print('Creative:', resp.content[0].text)

In [ ]:
# 2e. Streaming
print('Streaming: ', end='')
with model.chat(messages, system='Be concise.', max_tokens=100, stream=True) as stream:
    for text in stream.text_stream:
        print(text, end='', flush=True)
print()

## 3. Validate (Primary Use Case)

In [ ]:
# 3a. Safe proposal — should approve
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement from WS-042 to DC-01 via PsExec. Source 10.0.5.42.'
)
print('Safe proposal:')
print(resp.content[0].text)

In [ ]:
# 3b. Dangerous proposal — should reject
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.content[0].text)

In [ ]:
# 3c. Batch validate
proposals = [
    {'proposal': 'Block IP 10.0.5.12 at firewall.', 'context': 'Confirmed C2 from 10.0.5.12.'},
    {'proposal': 'Delete all firewall rules.', 'context': 'Minor config drift detected.'},
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.content[0].text[:120]}...')

## 4. Tool Calling (Anthropic Format)

In [ ]:
tools = [
    {
        'name': 'query_siem',
        'description': 'Search SIEM logs for security events',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Search query'},
                'time_range': {'type': 'string', 'description': 'Time range'},
            },
            'required': ['query']
        }
    },
    {
        'name': 'isolate_host',
        'description': 'Isolate a host from the network',
        'input_schema': {
            'type': 'object',
            'properties': {
                'hostname': {'type': 'string'},
                'reason': {'type': 'string'},
            },
            'required': ['hostname', 'reason']
        }
    }
]

In [ ]:
# 4a. Auto tool choice
tc_messages = [{'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12 in the last hour.'}]
resp = model.tool_call(tc_messages, tools, system='You are a SOC analyst.')
for block in resp.content:
    if block.type == 'tool_use':
        print(f'Tool call: {block.name}({block.input})')
    elif block.type == 'text':
        print(f'Text: {block.text[:80]}')

In [ ]:
# 4b. Required tool choice
resp = model.tool_call_required(tc_messages, tools)
for block in resp.content:
    if block.type == 'tool_use':
        print(f'Required: {block.name}({block.input})')

In [ ]:
# 4c. Specific tool
resp = model.tool_call_specific(tc_messages, tools, 'isolate_host')
for block in resp.content:
    if block.type == 'tool_use':
        print(f'Specific: {block.name}({block.input})')

## 5. Structured Output (JSON)

In [ ]:
json_messages = [
    {'role': 'user', 'content': 'Classify: "Multiple failed SSH logins from 10.0.5.12". Return {"severity": str, "category": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.content[0].text)

## 6. Batch Chat

In [ ]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.content[0].text}')

## 7. Token Usage

In [ ]:
resp = model.chat(messages, max_tokens=100)
print(f'Input tokens:  {resp.usage.input_tokens}')
print(f'Output tokens: {resp.usage.output_tokens}')

## 8. Get Config

In [ ]:
import json
print(json.dumps(model.get_config(), indent=2))

## Summary

All tests passed if no cells raised exceptions above.